## Points to improve
- ~smoothen slope using 5th and 95th percentiles~
- ~convert decibels to linear~
- ~test code in Kashmir (valley)~
- push low level code to utils and other relevant modules
- final checks on documentation

## Generated mean and sd for the following in this script:
- 44R
- 46R
- 47R
- 44Q
- 44P
- 42R
- 43R

## East India zones
'44P', '44Q', '44R', '45Q', '45R', '46Q', '46R', '47R'

In [1]:
import pyarrow
print("pyarrow:", pyarrow.__version__, "from", pyarrow.__file__)
import pyarrow.dataset as ds
print("dataset import OK")

pyarrow: 22.0.0 from C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\pyarrow\__init__.py
dataset import OK


In [2]:
import sys
sys.path.append('../')

from autofloods import flood_mapper
from autofloods import utils
import geopandas as gpd
import time

In [3]:
# small piece of code to get zone-wise list of IDs
gdf = gpd.read_file(r'../resources/india_utm_fishnet_buffer.gpkg')
zone_id_group = gdf[['zone', 'ID']].groupby('zone')['ID'].apply(list)

zone_id_dict = dict()

for idx in zone_id_group.index:
    zone_id_dict[idx] = zone_id_group[idx]

In [13]:
gdf['ID'].unique().shape

(380,)

In [16]:
import glob

for idx in gdf['ID'].unique():
    processed_files = glob.glob(f'../output/flood_raster/monthly*/*_{idx}_monthly.tif')
    
    print(f'{len(processed_files)} files found for ID: {idx}.')

2 files found for ID: 1.
2 files found for ID: 2.
2 files found for ID: 3.
2 files found for ID: 4.
2 files found for ID: 5.
2 files found for ID: 6.
2 files found for ID: 7.
2 files found for ID: 8.
2 files found for ID: 9.
2 files found for ID: 10.
2 files found for ID: 11.
2 files found for ID: 12.
2 files found for ID: 13.
2 files found for ID: 14.
2 files found for ID: 15.
2 files found for ID: 16.
2 files found for ID: 17.
2 files found for ID: 18.
2 files found for ID: 19.
2 files found for ID: 20.
2 files found for ID: 21.
2 files found for ID: 22.
2 files found for ID: 23.
2 files found for ID: 24.
2 files found for ID: 25.
2 files found for ID: 26.
2 files found for ID: 27.
2 files found for ID: 28.
2 files found for ID: 29.
2 files found for ID: 30.
2 files found for ID: 31.
2 files found for ID: 32.
2 files found for ID: 33.
2 files found for ID: 34.
2 files found for ID: 35.
2 files found for ID: 36.
2 files found for ID: 37.
2 files found for ID: 38.
2 files found for ID:

0 files found for ID: 317.
0 files found for ID: 318.
0 files found for ID: 319.
0 files found for ID: 320.
0 files found for ID: 321.
0 files found for ID: 322.
0 files found for ID: 323.
0 files found for ID: 324.
0 files found for ID: 325.
0 files found for ID: 326.
0 files found for ID: 327.
0 files found for ID: 328.
0 files found for ID: 329.
0 files found for ID: 330.
0 files found for ID: 331.
0 files found for ID: 332.
0 files found for ID: 333.
0 files found for ID: 334.
0 files found for ID: 335.
0 files found for ID: 336.
0 files found for ID: 337.
0 files found for ID: 338.
0 files found for ID: 339.
0 files found for ID: 340.
0 files found for ID: 341.
0 files found for ID: 342.
0 files found for ID: 343.
0 files found for ID: 344.
0 files found for ID: 345.
0 files found for ID: 346.
0 files found for ID: 347.
0 files found for ID: 348.
0 files found for ID: 349.
0 files found for ID: 350.
0 files found for ID: 351.
0 files found for ID: 352.
0 files found for ID: 353.
0

In [5]:
# 2025 leftovers: ['42Q', '42R', '43P', '43Q', '43R', '43S', '44R', '45R', '46Q', '46R']

In [6]:
%%time

#for wet_period in ['2017/07', '2021/07', '2022/07', '2023/07']:
for year in [2025]:
    for month in ['08', '09', '10']:
        wet_period = f'{year}/{month}'
        
        for n, zone_id in enumerate(['42Q', '42R', '43P', '43Q', '43R', '43S', '44R', '45R', '46Q', '46R']):#zone_id_dict.keys()):
            print(f'\nProcessing: {zone_id}. {n} out of {len(zone_id_dict)}.')
            t1 = time.time()

            #try:
            all_id_list = zone_id_dict[zone_id]
            all_id_list = [
                all_id_list[i:i+5]
                for i in range(0, len(all_id_list), 5)
            ]

            for id_list in all_id_list:
                print(f'Processing IDs: {id_list}')
                # create the flood mapper class
                flood_mapper_obj = flood_mapper(
                    grid_shapefile = r'../resources/india_utm_fishnet_buffer.gpkg',
                    grid_id_list = id_list, #zone_id_dict[zone_id],#[ID]
                    dry_date_col = 'dry_month',
                    id_col = 'ID',
                    dry_years=[2021, 2023],
                    slope_dir = r'../resources/slope/',
                    wet_duration = [wet_period, wet_period]
                )

                flood_mapper_obj.get_dry_dates()

                if len(flood_mapper_obj.aoi_ids_to_process) > 0:
                    flood_mapper_obj.generate_dry_date_ranges()
                    flood_mapper_obj.get_s1_items(dry_wet='dry')
                    flood_mapper_obj.read_scenes(dry_wet='dry', overview_level=2)
                    flood_mapper_obj.generate_mean_std_by_aoi()
                else:
                    flood_mapper_obj.load_mean_std_by_aoi()

                flood_mapper_obj.prepare_slope(dem_overview=0, buffer=500)

                flood_mapper_obj.prepare_wet_scenes(overview_level=2)
                flood_mapper_obj.generate_number_of_scenes(export_raster=True)
                flood_mapper_obj.map_floods(vv_thd=-2.5, vh_thd=-2.5, rel_slope_thd=20,
                                              export_raster=False, export_vector=True, export_maps=False)
                flood_mapper_obj.merge_floods_by_date(export_raster=True)
                flood_mapper_obj.monthly_sum()

            t2 = time.time()
            t_delta = t2 - t1
            print(f'Time taken for {zone_id} zone: {(t_delta / 60):.2f} mins.\n')
            #except:
            #    print(f'Some issue in this zone. Skipping..')


Processing: 42Q. 0 out of 15.
Processing IDs: [1, 2, 3, 4, 5]
Following previously processed AOI IDs found. Will be skipped. If you are running with a new year range, consider modifying or deleting the json file.
 []


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\scipy\interpolate\_interpolate.py:482: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\scipy\interpolate\_interpolate.py:479: RuntimeWarning: invalid value encountered in subtract
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\scipy\interpolate\_interpolate.py:479: RuntimeWarning: invalid value encountered in subtract
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\scipy\interpolate\_interpolate.py:482: RuntimeWarning

Slope for tile ID 1 found, will not be downloaded.
Slope for tile ID 2 found, will not be downloaded.
Slope for tile ID 3 found, will not be downloaded.
Slope for tile ID 4 found, will not be downloaded.
Slope for tile ID 5 found, will not be downloaded.


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\scipy\interpolate\_interpolate.py:482: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\scipy\interpolate\_interpolate.py:482: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


Processing IDs: [6, 7, 8, 9, 10]
Following previously processed AOI IDs found. Will be skipped. If you are running with a new year range, consider modifying or deleting the json file.
 []


...

Slope for tile ID 95 found, will not be downloaded.
Slope for tile ID 96 found, will not be downloaded.
Slope for tile ID 97 found, will not be downloaded.
Slope for tile ID 98 found, will not be downloaded.
Slope for tile ID 99 found, will not be downloaded.


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Processing IDs: [100]
Following previously processed AOI IDs found. Will be skipped. If you are running with a new year range, consider modifying or deleting the json file.
 []


C:\Users\ptripathy\AppData\Local\miniforge-pypy3\envs\autofloods\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


RasterioIOError: Read failed. See previous exception for details.

In [6]:
import shutil, glob

shutil.make_archive('../', 'zip', r'../ifmiap/')

'/home/pratyusht/datadrive/ifmiap/scripts.zip'

In [16]:
import shutil, glob

shutil.make_archive('../output/flood_monthlyadded_201907', 'zip', r'/home/pratyusht/datadrive/ifmiap/output/flood_raster/monthlyadded/',
                   *glob.glob(r'/home/pratyusht/datadrive/ifmiap/output/output/flood_raster/monthlyadded/*WET_201907_201907*')[:2])

'/home/pratyusht/datadrive/ifmiap/output/flood_monthlyadded_201907.zip'

In [ ]:
#!sudo shutdown now